# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² clinical colorectal cancer dataset using the [mlcroissant](https://pypi.org/project/mlcroissant/) library, following the Croissant schema and referencing all record sets, fields, and columns by their `@id`.

### Dataset Source
The dataset is provided by a FAIR² Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

It contains clinical and molecular features of 77 cancer survivors with second primary colorectal cancer.

In [ ]:
# Install the `mlcroissant` library if not already present
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and view the title & description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# For visualization later
import matplotlib.pyplot as plt
import seaborn as sns

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Let's review the dataset's available **record sets**, showing their `@id` and fields (columns). All elements are referenced by their `@id`.

This step lets us list all the available record sets, and for each, enumerate the fields and types. We'll then pick a record set for extraction (in this dataset, there is usually just one main tabular record set).

In [ ]:
# List all available record sets in the schema by @id
print("Available record sets in dataset:")
for record_set in metadata.record_sets:
    print(f"  RecordSet @id: {record_set.id}")
    if hasattr(record_set, 'name'):
        print(f"    name: {record_set.name}")
    # List the fields (columns)
    print("    Fields (by @id):")
    for field in record_set.fields:
        print(f"      - {field.id} ({getattr(field, 'data_type', 'unknown')})")
    print()

## 3. Data Extraction

We can extract tabular data from a **record set** using its `@id` (as seen above).

Below, we'll load all records for the principal record set found in this dataset. Adjust the code if there are multiple record sets or you're interested in a subset.

In [ ]:
# Extract data for ALL record sets
dataframes = {}
record_set_ids = [record_set.id for record_set in metadata.record_sets]

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  {len(df)} rows, columns: {df.columns.tolist()}")

# For downstream steps, select the main tabular record set
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
else:
    raise RuntimeError("No record sets found in the schema.")

Let's preview a few records (rows) from the main table. All columns shown reference their field `@id` (not display names).


In [ ]:
# Show field (column) @ids in the main table
print(f"Main RecordSet @id: {main_record_set_id}")
print("Field @ids (column names):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll:
- Select a numeric field by its `@id` (pick an age or count-like field as appropriate)
- Filter records based on this field
- Normalize values
- Group by a categorical attribute (such as sex or anatomical site)

⚠️ Please replace `<numeric_field_id>` and `<group_field_id>` below with the actual field `@id` from your dataset (chosen from above). For demonstration, we'll try to auto-detect an integer or float column if possible.

In [ ]:
# Identify a likely numeric (int/float) field by checking data types or field_id substrings
df = dataframes[main_record_set_id]
# Attempt to find a numeric column by type
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
# Fallback: look for common field ids (replace as appropriate for new datasets)
common_numeric_ids = ['schema:age', 'cr:age', 'schema:Age', 'cr:interval_between_diagnoses', 'cr:interval_in_months', 'cr:diagnosis_interval']
candidate = None
for col in numeric_field_candidates:
    candidate = col
    break
if candidate is None:
    for col in df.columns:
        if any(cid in col for cid in common_numeric_ids):
            candidate = col
            break

if candidate is None:
    raise RuntimeError("No numeric field detected in the record set. Please specify its @id manually.")

numeric_field_id = candidate

print(f"Using numeric field: {numeric_field_id}")
# Choose an arbitrary threshold for a demonstration; adjust as needed.
if df[numeric_field_id].dtype.kind in 'iufc':
    threshold = df[numeric_field_id].mean()
else:
    threshold = 10

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} ({len(filtered_df)} rows):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Pick a group variable (categorical) from known ids or by dtype
group_field = None
likely_group_ids = ['schema:sex', 'schema:gender', 'cr:sex', 'cr:site', 'cr:anatomical_location', 'cr:MSI_status', 'cr:status', 'cr:group']
# Try to find categorical columns present
for col in df.columns:
    if df[col].dtype == object or df[col].dtype.name == 'category':
        if any(substr in col for substr in likely_group_ids):
            group_field = col
            break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

We now visualize the distribution of the selected numeric field, and if grouped, show group-wise averages. Visualization references all variables by their `@id`.

In [ ]:
# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Grouped bar plot if group_field available
if group_field:
    plt.figure(figsize=(8, 5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id, palette='tab10')
    plt.title(f'Mean {numeric_field_id} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded clinical and molecular data from the FAIR² colorectal cancer dataset using the Croissant schema and `mlcroissant`
- Explored available record sets, fields, and their `@id`s for robust referencing
- Extracted records from the main tabular record set
- Performed basic EDA: filtering, normalization, and grouping by clinical attributes
- Visualized numeric distributions and group means

**All fields and subsets were always referenced by their Croissant `@id` as required.**

Continue your analysis by exploring more complex relationships, using clinical outcomes or molecular subtypes as groupings, or applying ML models referencing fields by `@id`.